# COMP3314 Image Classification Pipeline (Wavelet Scattering + Classical ML)

This notebook builds a non-deep-learning image classifier for the COMP3314 challenge dataset (CIFAR-10 style, 10 classes).
The core idea is simple: use a fixed feature extractor, then let classical ML do the classification.

## Pipeline

1. Load data from the challenge CSV+image format.
2. **Extract wavelet scattering features** in batches (GPU when available).
3. **Cache features to disk** so we can resume safely after OOM/interruption.
4. **Scale + reduce dimensions (PCA)** using train-set statistics only.
5. **Train classifier** (RBF SVM first, Logistic Regression fallback).
6. **Run test-time augmentation (horizontal flip TTA)** and average scores.
7. **Export submission.csv** in `im_name,label` format.

## Why this design?

- **Scattering features** are stable and strong without backprop training.
- **Classical ML** is lighter than full CNN training and often easier to tune quickly.
- **Caching + cleanup** makes the notebook practical on shared GPU servers with memory limits.
- **CPU/GPU fallback paths** keep the pipeline runnable even when RAPIDS is unavailable.

Hence this notebook provides a reproducible, restart-friendly workflow. Most heavy steps are in feature extraction + PCA/SVM fitting, so make sure to monitor memory usage and adjust batch sizes accordingly to prevent OOM errors if there are any.

## 1) Environment and Reproducibility

This section imports all dependencies and sets up a **GPU-first but not GPU-only** workflow.

- If RAPIDS cuML is available, PCA / scaler / classifier run on GPU.
- If cuML is missing, code automatically falls back to scikit-learn on CPU.
- If CuPy is missing, array ops still run with NumPy.

So even on a normal laptop the notebook can still run (just slower). Because of this, we highly recommend to configure a conda RAPIDS environment with cuML for best performance. You can refer to the RAPIDS installation guide for instructions: https://rapids.ai/start.html

### Expected data layout

Inside `./data`, make sure these exist:

- `train.csv` and `test.csv`
- `train_ims/` and `test_ims/`

`test.csv` must contain `im_name` for final submission mapping.

In [ ]:
# Core imports
import os
import gc
import warnings
import numpy as np
import pandas as pd
import torch
from torchvision import transforms
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler as SKStandardScaler
from sklearn.decomposition import PCA as SKPCA
from sklearn.svm import SVC as SKSVC
from sklearn.linear_model import LogisticRegression as SKLogisticRegression

# Wavelet Scattering Transform
from kymatio.torch import Scattering2D

# RAPIDS cuML
HAS_CUML = True
try:
    import cuml
    from cuml.decomposition import PCA
    from cuml.svm import SVC
    from cuml.linear_model import LogisticRegression
    from cuml.preprocessing import StandardScaler
except Exception as cuml_error:
    HAS_CUML = False
    cuml = None
    from sklearn.decomposition import PCA
    from sklearn.svm import SVC
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    print(f"cuML import failed, falling back to scikit-learn. Details: {cuml_error}")

# CuPy for GPU array handling
try:
    import cupy as cp
except ImportError:
    cp = None
    print("cupy not available. Will fall back to NumPy on CPU.")

warnings.filterwarnings('ignore')
print(f"All imports successful. Backend: {'cuML (GPU)' if HAS_CUML else 'scikit-learn (CPU)'}")

All imports successful. Backend: cuML (GPU)


We now fix random seeds across NumPy / PyTorch / CuPy to reduce run-to-run variance and make experiments easier to compare.

In [2]:
# Fixed seed for deterministic behavior
SEED = 42

# NumPy
np.random.seed(SEED)

# PyTorch
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

# CuPy (if available)
if cp is not None:
    cp.random.seed(SEED)

print(f"Random seeds set to {SEED} for reproducibility.")

Random seeds set to 42 for reproducibility.


Before touching data, we print device info (CUDA, GPU count, memory, CuPy status). This is a quick sanity check so we know whether we are actually running GPU or CPU path.

In [3]:
# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch Device: {device}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"Current GPU Memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated")
else:
    print("No CUDA GPU detected. Pipeline will run on CPU.")

# Check CuPy
if cp is not None:
    print(f"CuPy available: {cp.__version__}")
else:
    print("CuPy not available.")

PyTorch Device: cuda
CUDA Version: 12.9
GPU Count: 2
  GPU 0: NVIDIA GeForce RTX 4080 SUPER
  GPU 1: NVIDIA GeForce RTX 4080 SUPER
Current GPU Memory: 0.00 GB allocated
CuPy available: 14.0.1


## 2) Data Path Resolution

This block auto-detects project root and verifies required files exist.
It supports two common notebook launch locations (workspace root vs project folder), so you don't need to manually `cd` every time.

It also creates a `cache/` directory used later for saved scattering batches.

In [4]:
from pathlib import Path

# Resolve project root robustly (works even if notebook kernel cwd is workspace root)
cwd = Path.cwd()
candidate_roots = [
    cwd,
    cwd / 'COMP3314-ML-Challenge',
]

PROJECT_ROOT = None
for root in candidate_roots:
    if (root / 'data' / 'train.csv').exists() and (root / 'data' / 'test.csv').exists():
        PROJECT_ROOT = root.resolve()
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Could not locate project root from cwd={cwd}. Expected data/train.csv and data/test.csv."
    )

DATA_ROOT = str(PROJECT_ROOT / 'data')
CACHE_DIR = str(PROJECT_ROOT / 'cache')

# Ensure cache directory exists
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data root: {DATA_ROOT}")
print(f"Cache directory: {CACHE_DIR}")
print(f"Data directory contents: {os.listdir(DATA_ROOT) if os.path.exists(DATA_ROOT) else 'empty'}")

Project root: /userhome/cs/u3645267/COMP3314-ML-Challenge
Data root: /userhome/cs/u3645267/COMP3314-ML-Challenge/data
Cache directory: /userhome/cs/u3645267/COMP3314-ML-Challenge/cache
Data directory contents: ['train.csv', 'test.csv', 'test_ims', 'train_ims']


## 3) Dataset Loading 

This notebook uses the challenge dataset format:

- `train.csv` + `train_ims/`
- `test.csv` + `test_ims/`

A custom Dataset class maps `im_name -> image file` and returns `(image_tensor, label)`.
This keeps the training/inference pipeline explicit and avoids extra branching logic.

In [ ]:
# No data augmentation for train set during loading
# (normalization and other transforms will be applied in feature extraction phase)
base_transform = transforms.ToTensor()

CIFAR10_CLASS_NAMES = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
 ]

print("Loading challenge CSV+image structure from train.csv/test.csv...")
from PIL import Image

class LocalCIFARLikeDataset(torch.utils.data.Dataset):
    def __init__(self, csv_path: str, image_dir: str, transform=None, class_names=None):
        self.df = pd.read_csv(csv_path)
        self.image_dir = image_dir
        self.transform = transform

        if 'im_name' not in self.df.columns:
            raise ValueError(f"{csv_path} must contain column 'im_name'")

        self.has_label = 'label' in self.df.columns
        if self.has_label:
            self.labels = self.df['label'].astype(np.int64).to_numpy()
            inferred_num_classes = int(self.labels.max()) + 1 if len(self.labels) > 0 else 10
        else:
            self.labels = np.full(len(self.df), -1, dtype=np.int64)
            inferred_num_classes = 10

        if class_names is not None and len(class_names) >= inferred_num_classes:
            self.classes = class_names[:inferred_num_classes]
        else:
            self.classes = [str(i) for i in range(inferred_num_classes)]

        self.targets = self.labels.tolist()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_name = self.df.iloc[idx]['im_name']
        image_path = os.path.join(self.image_dir, image_name)
        image = Image.open(image_path).convert('RGB')

        if self.transform is not None:
            image = self.transform(image)

        label = int(self.labels[idx])
        return image, label

train_csv = os.path.join(DATA_ROOT, 'train.csv')
test_csv = os.path.join(DATA_ROOT, 'test.csv')
train_dir = os.path.join(DATA_ROOT, 'train_ims')
test_dir = os.path.join(DATA_ROOT, 'test_ims')

for required_path in [train_csv, test_csv, train_dir, test_dir]:
    if not os.path.exists(required_path):
        raise FileNotFoundError(f"Required local data path not found: {required_path}")

cifar10_train = LocalCIFARLikeDataset(
    csv_path=train_csv,
    image_dir=train_dir,
    transform=base_transform,
    class_names=CIFAR10_CLASS_NAMES,
)
cifar10_test = LocalCIFARLikeDataset(
    csv_path=test_csv,
    image_dir=test_dir,
    transform=base_transform,
    class_names=CIFAR10_CLASS_NAMES,
)

print(f"  Training set loaded: {len(cifar10_train)} samples")
print(f"  Test set loaded: {len(cifar10_test)} samples")

Detected challenge CSV+image structure. Loading local dataset from train.csv/test.csv...
  Training set loaded: 50000 samples
  Test set loaded: 10000 samples


### Quick Sanity Check

We inspect class names, sample tensor shape, value range, and label distribution.
If anything looks wrong here (e.g., shape mismatch, weird label counts), fix data issues now before expensive feature extraction.

In [6]:
# Class names for CIFAR-10
CLASS_NAMES = cifar10_train.classes
NUM_CLASSES = len(CLASS_NAMES)

print(f"Number of classes: {NUM_CLASSES}")
print(f"Class names: {CLASS_NAMES}")

# Sample a batch to inspect shapes
sample_image, sample_label = cifar10_train[0]
print(f"\nSample image shape: {sample_image.shape}")
print(f"Sample label: {sample_label} ({CLASS_NAMES[sample_label]})")
print(f"Image dtype: {sample_image.dtype}")
print(f"Image value range: [{sample_image.min():.3f}, {sample_image.max():.3f}]")

# Label distribution in training set
train_labels = np.array([label for _, label in cifar10_train])
print(f"\nTraining set label distribution:")
for class_idx, class_name in enumerate(CLASS_NAMES):
    count = np.sum(train_labels == class_idx)
    print(f"  {class_name}: {count} samples")

Number of classes: 10
Class names: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

Sample image shape: torch.Size([3, 32, 32])
Sample label: 9 (truck)
Image dtype: torch.float32
Image value range: [0.043, 0.976]



Training set label distribution:
  airplane: 5027 samples
  automobile: 5008 samples
  bird: 5026 samples
  cat: 4946 samples
  deer: 5061 samples
  dog: 5005 samples
  frog: 4962 samples
  horse: 5006 samples
  ship: 5012 samples
  truck: 4947 samples


## 4) Scattering Feature Extraction and Disk Caching

This is the most memory-sensitive stage.
We compute wavelet scattering features in mini-batches and save each batch as `.npy` files.

### Why caching?

Running the notebook from start every time you encounter an OOM error is painful af. This avoid re-running expensive extraction after crash/interruption. The notebook will resume from existing batches when `force_reextract=False`
Which also keeps GPU peak memory lower by batch processing + explicit cleanup.

Output files include:

- `train_batch_k.npy` (+ labels)
- `test_batch_k.npy`

These cached arrays become the input for model training in the next section.

In [7]:
# Conservative extraction configuration
BATCH_SIZE = 128
SCATTERING_J = 2
SCATTERING_SHAPE = (32, 32)
EPSILON = 1e-8
FORCE_REEXTRACT = True  # Set False if you intentionally want to reuse old cache files

# Keep workers low to avoid host RAM pressure on shared machines
NUM_WORKERS = 2 if torch.cuda.is_available() else 0

# Initialize fixed (non-trainable) wavelet feature extractor
scattering = Scattering2D(J=SCATTERING_J, shape=SCATTERING_SHAPE).to(device)

print(f"Scattering2D ready: J={SCATTERING_J}, shape={SCATTERING_SHAPE}, batch_size={BATCH_SIZE}")
print(f"Force re-extract cache: {FORCE_REEXTRACT}")

Scattering2D ready: J=2, shape=(32, 32), batch_size=128
Force re-extract cache: True


In [ ]:
def extract_and_cache_scattering_batches(
    dataset,
    split_name: str,
    scattering_module,
    batch_size: int = BATCH_SIZE,
    force_reextract: bool = FORCE_REEXTRACT,
):
    """Extract scattering features in resumable batches and cache to disk."""
    loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    processed_batches = 0
    skipped_batches = 0

    for batch_idx, (images, labels) in enumerate(loader):
        feature_path = os.path.join(CACHE_DIR, f"{split_name}_batch_{batch_idx}.npy")
        label_path = os.path.join(CACHE_DIR, f"{split_name}_labels_batch_{batch_idx}.npy")

        if split_name == 'train':
            batch_cached = os.path.exists(feature_path) and os.path.exists(label_path)
        else:
            batch_cached = os.path.exists(feature_path)

        if batch_cached and not force_reextract:
            skipped_batches += 1
            continue

        with torch.no_grad():
            images = images.to(device, non_blocking=torch.cuda.is_available())
            scattering_features = scattering_module(images)
            scattering_features = scattering_features.reshape(scattering_features.shape[0], -1)
            batch_features = scattering_features.detach().cpu().numpy().astype(np.float32, copy=False)

        # Important: keep raw scattering features here. Global scaler will be fitted on train set only later.
        np.save(feature_path, batch_features)

        if split_name == 'train':
            np.save(label_path, labels.numpy().astype(np.int64, copy=False))

        processed_batches += 1

        # In-loop cleanup to reduce peak memory usage
        del images, labels, scattering_features, batch_features
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print(
        f"{split_name}: total={len(loader)} | processed={processed_batches} | skipped_existing={skipped_batches}"
    )

    del loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [9]:
# Run extraction with resumable caching behavior
extract_and_cache_scattering_batches(cifar10_train, split_name='train', scattering_module=scattering, batch_size=BATCH_SIZE)
extract_and_cache_scattering_batches(cifar10_test, split_name='test', scattering_module=scattering, batch_size=BATCH_SIZE)

train: total=391 | processed=391 | skipped_existing=0
test: total=79 | processed=79 | skipped_existing=0


In [ ]:
# Cleanup
del scattering
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Feature extraction + caching complete. Stage memory cleanup done.")

Feature extraction + caching complete. Stage memory cleanup done.


## 5) Classifier Training and Inference

Now we switch from image space to cached feature space.
Main steps:

1. Load cached train/test features in deterministic batch order
2. Fit **global scaler on train only**, then transform train+test
3. Run CV search on a stratified subset to pick PCA components and SVM hyperparameters
4. Fit final PCA on full training data
5. Train final classifier (SVM preferred, logistic regression fallback)

This keeps tuning cost manageable while still using full data for final fit.

In [11]:
import re
from pathlib import Path


def _discover_cache_dir() -> str:
    """Auto-detect the cache directory containing extracted scattering batches."""
    candidate_dirs = []

    # 1) Configured cache directory (if defined)
    if 'CACHE_DIR' in globals() and CACHE_DIR is not None:
        candidate_dirs.append(Path(CACHE_DIR))

    # 2) Common fallbacks (kernel cwd may differ from project root)
    cwd = Path.cwd()
    candidate_dirs.extend([
        cwd / 'cache',
        cwd / 'COMP3314-ML-Challenge' / 'cache',
    ])

    # Deduplicate while preserving order
    seen = set()
    unique_dirs = []
    for path in candidate_dirs:
        resolved = path.resolve()
        if resolved not in seen:
            seen.add(resolved)
            unique_dirs.append(resolved)

    def batch_count(cache_path: Path) -> int:
        if not cache_path.exists() or not cache_path.is_dir():
            return 0
        return len(list(cache_path.glob('train_batch_*.npy')))

    scored = [(batch_count(path), path) for path in unique_dirs]
    best_count, best_path = max(scored, key=lambda item: item[0]) if scored else (0, None)

    if best_path is None or best_count == 0:
        raise FileNotFoundError(
            f"No train cache files found. Checked: {[str(p) for p in unique_dirs]}"
        )

    print(f"Using cache directory: {best_path} (train batches found: {best_count})")
    return str(best_path)


EFFECTIVE_CACHE_DIR = _discover_cache_dir()


def _sorted_cache_entries(prefix: str):
    """Return (batch_idx, filename) pairs sorted by batch index."""
    pattern = re.compile(rf"^{re.escape(prefix)}(\d+)\.npy$")
    entries = []

    for filename in os.listdir(EFFECTIVE_CACHE_DIR):
        match = pattern.match(filename)
        if match:
            entries.append((int(match.group(1)), filename))

    entries.sort(key=lambda item: item[0])
    return entries


def load_cached_train_data():
    """Load cached train features/labels in deterministic batch order."""
    feature_entries = _sorted_cache_entries("train_batch_")
    label_entries = _sorted_cache_entries("train_labels_batch_")

    if not feature_entries or not label_entries:
        raise FileNotFoundError(
            f"Missing cached train features or labels in {EFFECTIVE_CACHE_DIR}"
        )

    if len(feature_entries) != len(label_entries):
        raise ValueError(
            f"Train cache count mismatch: features={len(feature_entries)}, labels={len(label_entries)}"
        )

    feature_batches = []
    label_batches = []

    for (f_idx, f_name), (l_idx, l_name) in zip(feature_entries, label_entries):
        if f_idx != l_idx:
            raise ValueError(
                f"Train batch index misalignment: feature_batch={f_idx}, label_batch={l_idx}"
            )

        x_batch = np.load(os.path.join(EFFECTIVE_CACHE_DIR, f_name)).astype(np.float32, copy=False)
        y_batch = np.load(os.path.join(EFFECTIVE_CACHE_DIR, l_name)).astype(np.int64, copy=False)

        if x_batch.shape[0] != y_batch.shape[0]:
            raise ValueError(
                f"Row mismatch in batch {f_idx}: features={x_batch.shape[0]}, labels={y_batch.shape[0]}"
            )

        feature_batches.append(x_batch)
        label_batches.append(y_batch)

    x_train = np.concatenate(feature_batches, axis=0).astype(np.float32, copy=False)
    y_train = np.concatenate(label_batches, axis=0).astype(np.int64, copy=False)

    del feature_batches, label_batches
    gc.collect()

    return x_train, y_train


def load_cached_test_features():
    """Load cached test features in deterministic batch order."""
    test_entries = _sorted_cache_entries("test_batch_")

    if not test_entries:
        raise FileNotFoundError(
            f"Missing cached test features in {EFFECTIVE_CACHE_DIR}"
        )

    test_batches = []
    for _, f_name in test_entries:
        x_batch = np.load(os.path.join(EFFECTIVE_CACHE_DIR, f_name)).astype(np.float32, copy=False)
        test_batches.append(x_batch)

    x_test = np.concatenate(test_batches, axis=0).astype(np.float32, copy=False)

    del test_batches
    gc.collect()

    return x_test


X_train_cached, y_train_cached = load_cached_train_data()
X_test_cached = load_cached_test_features()

print(f"Loaded cached train features: {X_train_cached.shape}, dtype={X_train_cached.dtype}")
print(f"Loaded cached train labels: {y_train_cached.shape}, dtype={y_train_cached.dtype}")
print(f"Loaded cached test features: {X_test_cached.shape}, dtype={X_test_cached.dtype}")

Using cache directory: /userhome/cs/u3645267/COMP3314-ML-Challenge/cache (train batches found: 391)
Loaded cached train features: (50000, 15552), dtype=float32
Loaded cached train labels: (50000,), dtype=int64
Loaded cached test features: (10000, 15552), dtype=float32


### 5.1 Cache Loading Notes

The loader validates batch alignment strictly (feature batch index must match label batch index).
If cache files are incomplete or mismatched, it raises clear errors early, which is much safer than silently training on broken data.

At the end of this block, `X_train_cached`, `y_train_cached`, and `X_test_cached` are the canonical inputs for all later training steps.

In [ ]:
SEARCH_ROWS = 12000
N_SPLITS = 3
N_COMPONENTS_GRID = [768, 1024, 1536]
C_GRID = [5.0, 8.0, 10.0, 12.0]
GAMMA_GRID = ['scale']

print("Fitting global scaler on training features only...")
if HAS_CUML and cp is not None:
    scaler = StandardScaler(with_mean=True, with_std=True)
    X_train_scaled = scaler.fit_transform(cp.asarray(X_train_cached, dtype=cp.float32))
    X_test_scaled = scaler.transform(cp.asarray(X_test_cached, dtype=cp.float32))
    backend_name = "cuML + CuPy"
elif HAS_CUML:
    scaler = StandardScaler(with_mean=True, with_std=True)
    X_train_scaled = scaler.fit_transform(X_train_cached)
    X_test_scaled = scaler.transform(X_test_cached)
    backend_name = "cuML + NumPy"
else:
    scaler = StandardScaler(with_mean=True, with_std=True)
    X_train_scaled = scaler.fit_transform(X_train_cached)
    X_test_scaled = scaler.transform(X_test_cached)
    backend_name = "scikit-learn CPU"

print(f"Scaler backend: {backend_name}")
print(f"Scaled train shape: {X_train_scaled.shape}, test shape: {X_test_scaled.shape}")

# Stratified subset for practical search cost
search_size = min(SEARCH_ROWS, y_train_cached.shape[0])
rng = np.random.default_rng(SEED)
if search_size < y_train_cached.shape[0]:
    per_class = max(1, search_size // len(np.unique(y_train_cached)))
    selected = []
    for class_id in np.unique(y_train_cached):
        class_idx = np.where(y_train_cached == class_id)[0]
        take = min(per_class, class_idx.shape[0])
        selected.append(rng.choice(class_idx, size=take, replace=False))
    search_idx = np.concatenate(selected)
    if search_idx.shape[0] > search_size:
        search_idx = rng.choice(search_idx, size=search_size, replace=False)
else:
    search_idx = np.arange(y_train_cached.shape[0])

search_idx = np.sort(search_idx)
X_search_cpu = X_train_cached[search_idx].astype(np.float32, copy=False)
y_search = y_train_cached[search_idx]

print(f"Search subset rows: {len(search_idx)}")
print("Joint search backend: scikit-learn CPU proxy (for stability), final fit remains GPU-first.")

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
search_records = []
best_score = -1.0
best_cfg = None

for n_components in N_COMPONENTS_GRID:
    for c_val in C_GRID:
        for gamma_val in GAMMA_GRID:
            fold_scores = []

            for tr_idx, va_idx in skf.split(np.zeros_like(y_search), y_search):
                X_tr = X_search_cpu[tr_idx]
                X_va = X_search_cpu[va_idx]
                y_tr = y_search[tr_idx]
                y_va = y_search[va_idx]

                scaler_fold = SKStandardScaler(with_mean=True, with_std=True)
                X_tr_s = scaler_fold.fit_transform(X_tr)
                X_va_s = scaler_fold.transform(X_va)

                pca_fold = SKPCA(n_components=n_components)
                X_tr_pca = pca_fold.fit_transform(X_tr_s)
                X_va_pca = pca_fold.transform(X_va_s)

                try:
                    clf_fold = SKSVC(kernel='rbf', C=c_val, gamma=gamma_val)
                    clf_fold.fit(X_tr_pca, y_tr)
                    va_pred = clf_fold.predict(X_va_pca)
                except Exception:
                    clf_fold = SKLogisticRegression(max_iter=1000, random_state=SEED)
                    clf_fold.fit(X_tr_pca, y_tr)
                    va_pred = clf_fold.predict(X_va_pca)

                fold_acc = float((va_pred.reshape(-1) == y_va).mean())
                fold_scores.append(fold_acc)

                del X_tr, X_va, X_tr_s, X_va_s, X_tr_pca, X_va_pca, va_pred
                gc.collect()

            cv_mean = float(np.mean(fold_scores))
            cv_std = float(np.std(fold_scores))
            robust_score = cv_mean - 0.5 * cv_std

            row = {
                'n_components': n_components,
                'C': c_val,
                'gamma': gamma_val,
                'cv_mean': cv_mean,
                'cv_std': cv_std,
                'robust_score': robust_score,
            }
            search_records.append(row)

            print(
                f"n_comp={n_components:4d} | C={c_val:>4} | gamma={gamma_val:<5} "
                f"| mean={cv_mean:.5f} | std={cv_std:.5f} | robust={robust_score:.5f}"
            )

            if robust_score > best_score:
                best_score = robust_score
                best_cfg = row

print("\nBest config (by mean - 0.5*std):", best_cfg)
search_results_df = pd.DataFrame(search_records).sort_values('robust_score', ascending=False)

BEST_N_COMPONENTS = int(best_cfg['n_components'])
BEST_C = float(best_cfg['C'])
BEST_GAMMA = best_cfg['gamma']

# Fit final PCA on full scaled train data (GPU-first)
if HAS_CUML:
    if cp is not None and not isinstance(X_train_scaled, np.ndarray):
        pca = PCA(n_components=BEST_N_COMPONENTS, output_type='cupy')
        print("Using cuML PCA with CuPy arrays for final fit.")
    else:
        pca = PCA(n_components=BEST_N_COMPONENTS, output_type='numpy')
        print("Using cuML PCA with NumPy arrays for final fit.")
else:
    pca = PCA(n_components=BEST_N_COMPONENTS)
    print("Using scikit-learn PCA on CPU for final fit.")

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"Final PCA complete: train -> {X_train_pca.shape}, test -> {X_test_pca.shape}")
print(f"Selected params: n_components={BEST_N_COMPONENTS}, C={BEST_C}, gamma={BEST_GAMMA}")

if cp is not None and hasattr(X_train_pca, 'shape') and not isinstance(X_train_pca, np.ndarray):
    print("Final PCA arrays are on GPU (CuPy).")
else:
    print("Final PCA arrays are on CPU (NumPy).")

del X_search_cpu
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Fitting global scaler on training features only...
Scaler backend: cuML + CuPy
Scaled train shape: (50000, 15552), test shape: (10000, 15552)
Search subset rows: 12000
Joint search backend: scikit-learn CPU proxy (for stability), final fit remains GPU-first.
n_comp= 768 | C= 5.0 | gamma=scale | mean=0.65825 | std=0.00779 | robust=0.65436
n_comp= 768 | C= 8.0 | gamma=scale | mean=0.65917 | std=0.00893 | robust=0.65470
n_comp= 768 | C=10.0 | gamma=scale | mean=0.65758 | std=0.00700 | robust=0.65409
n_comp= 768 | C=12.0 | gamma=scale | mean=0.65750 | std=0.00857 | robust=0.65321
n_comp=1024 | C= 5.0 | gamma=scale | mean=0.66058 | std=0.00892 | robust=0.65612
n_comp=1024 | C= 8.0 | gamma=scale | mean=0.65950 | std=0.00796 | robust=0.65552
n_comp=1024 | C=10.0 | gamma=scale | mean=0.65967 | std=0.00840 | robust=0.65547
n_comp=1024 | C=12.0 | gamma=scale | mean=0.65942 | std=0.00807 | robust=0.65538
n_comp=1536 | C= 5.0 | gamma=scale | mean=0.66017 | std=0.00865 | robust=0.65584
n_comp=1536 

### 5.2 Model Search, Final Fit, and TTA

This part does a practical hyperparameter search on a stratified subset (`SEARCH_ROWS`) to save time.
Selection criterion is a robust score: `mean_cv - 0.5 * std_cv` (prefers stable configs, not only lucky peaks).

After selecting best params, we train on full data and run **horizontal flip TTA** on test images.
Final prediction uses average score from original view and flipped view, then writes `submission.csv`.

This gives a small robustness boost with minimal code complexity.

In [ ]:
def _to_numpy(x):
    if cp is not None and hasattr(x, 'get'):
        return cp.asnumpy(x)
    return np.asarray(x)


def _normalize_scores(scores: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    scores = scores.astype(np.float64, copy=False)
    scores = scores - scores.max(axis=1, keepdims=True)
    exp_scores = np.exp(scores)
    return exp_scores / (exp_scores.sum(axis=1, keepdims=True) + eps)


def _get_class_scores(model, x_input, num_classes: int):
    if hasattr(model, 'predict_proba'):
        try:
            probs = model.predict_proba(x_input)
            probs = _to_numpy(probs)
            if probs.ndim == 2 and probs.shape[1] == num_classes:
                return probs.astype(np.float64, copy=False), 'predict_proba'
        except Exception:
            pass

    if hasattr(model, 'decision_function'):
        try:
            decision = model.decision_function(x_input)
            decision = _to_numpy(decision)
            if decision.ndim == 1:
                one_hot = np.zeros((decision.shape[0], num_classes), dtype=np.float64)
                clipped = np.clip(decision.astype(np.int64, copy=False), 0, num_classes - 1)
                one_hot[np.arange(decision.shape[0]), clipped] = 1.0
                return one_hot, 'decision_function_1d_fallback'
            if decision.ndim == 2 and decision.shape[1] == num_classes:
                return _normalize_scores(decision), 'decision_function_softmax'
        except Exception:
            pass

    pred = model.predict(x_input)
    pred = _to_numpy(pred).astype(np.int64, copy=False).reshape(-1)
    one_hot = np.zeros((pred.shape[0], num_classes), dtype=np.float64)
    one_hot[np.arange(pred.shape[0]), pred] = 1.0
    return one_hot, 'predict_onehot_fallback'


gamma_for_fit = BEST_GAMMA if BEST_GAMMA in ('scale', 'auto') else float(BEST_GAMMA)

try:
    if HAS_CUML:
        classifier = SVC(kernel='rbf', C=BEST_C, gamma=gamma_for_fit)
        classifier.fit(X_train_pca, y_train_cached)
        classifier_name = f"cuml.svm.SVC(kernel=rbf, C={BEST_C}, gamma={gamma_for_fit})"
    else:
        classifier = SVC(kernel='rbf', C=BEST_C, gamma=gamma_for_fit)
        classifier.fit(X_train_pca, y_train_cached)
        classifier_name = f"sklearn.svm.SVC(kernel=rbf, C={BEST_C}, gamma={gamma_for_fit})"
except Exception as svc_error:
    if HAS_CUML:
        print(f"cuML SVC failed ({svc_error}). Falling back to cuML LogisticRegression.")
        classifier = LogisticRegression(max_iter=1000)
        classifier.fit(X_train_pca, y_train_cached)
        classifier_name = 'cuml.linear_model.LogisticRegression(max_iter=1000)'
    else:
        print(f"sklearn SVC failed ({svc_error}). Falling back to sklearn LogisticRegression.")
        classifier = LogisticRegression(max_iter=1000, random_state=SEED)
        classifier.fit(X_train_pca, y_train_cached)
        classifier_name = 'sklearn.linear_model.LogisticRegression(max_iter=1000)'

num_classes = int(np.max(y_train_cached)) + 1

# Base prediction scores on original cached test features
base_scores, base_score_source = _get_class_scores(classifier, X_test_pca, num_classes=num_classes)
base_predictions = base_scores.argmax(axis=1).astype(np.int64, copy=False)

# Flip-TTA: extract scattering features from horizontally flipped test images,
# then apply the SAME scaler and PCA fitted on training data.

print("Running horizontal flip TTA feature extraction...")
scattering_tta = Scattering2D(J=SCATTERING_J, shape=SCATTERING_SHAPE).to(device)

tta_loader = torch.utils.data.DataLoader(
    cifar10_test,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

tta_feature_batches = []
with torch.no_grad():
    for images, _ in tta_loader:
        images = images.to(device, non_blocking=torch.cuda.is_available())
        images_flipped = torch.flip(images, dims=[3])

        scattering_features = scattering_tta(images_flipped)
        scattering_features = scattering_features.reshape(scattering_features.shape[0], -1)
        batch_features = scattering_features.detach().cpu().numpy().astype(np.float32, copy=False)
        tta_feature_batches.append(batch_features)

        del images, images_flipped, scattering_features, batch_features
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

X_test_tta = np.concatenate(tta_feature_batches, axis=0).astype(np.float32, copy=False)

# Apply train-fitted scaler + PCA
if cp is not None and hasattr(X_test_scaled, 'shape') and not isinstance(X_test_scaled, np.ndarray):
    X_test_tta_scaled = scaler.transform(cp.asarray(X_test_tta, dtype=cp.float32))
else:
    X_test_tta_scaled = scaler.transform(X_test_tta)

X_test_tta_pca = pca.transform(X_test_tta_scaled)
tta_scores, tta_score_source = _get_class_scores(classifier, X_test_tta_pca, num_classes=num_classes)
tta_predictions = tta_scores.argmax(axis=1).astype(np.int64, copy=False)

# Two-view score averaging
final_scores = 0.5 * base_scores + 0.5 * tta_scores
predictions = final_scores.argmax(axis=1).astype(np.int64, copy=False)

if predictions.shape[0] != X_test_cached.shape[0]:
    raise ValueError(
        f"Prediction length mismatch: expected {X_test_cached.shape[0]}, got {predictions.shape[0]}"
    )

# Build submission using required challenge format: im_name,label
test_manifest_path = os.path.join(DATA_ROOT, 'test.csv')
test_manifest = pd.read_csv(test_manifest_path)
if 'im_name' not in test_manifest.columns:
    raise ValueError(f"{test_manifest_path} must contain column 'im_name'")
if len(test_manifest) != predictions.shape[0]:
    raise ValueError(
        f"Test manifest row mismatch: test.csv={len(test_manifest)}, predictions={predictions.shape[0]}"
    )

submission = pd.DataFrame({
    'im_name': test_manifest['im_name'].astype(str).to_numpy(),
    'label': predictions,
})
submission_path = './submission.csv'
submission.to_csv(submission_path, index=False)

agreement = float((base_predictions == tta_predictions).mean())
print(f"Classifier used: {classifier_name}")
print(f"Base score source: {base_score_source} | TTA score source: {tta_score_source}")
print(f"TTA agreement with base predictions: {agreement:.4f}")
print(f"Predictions generated: {predictions.shape[0]} rows")
print(f"Saved submission: {submission_path}")
print(f"Submission columns: {list(submission.columns)}")

del scattering_tta, tta_loader, tta_feature_batches, X_test_tta, X_test_tta_scaled, X_test_tta_pca
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Running horizontal flip TTA feature extraction...
Classifier used: cuml.svm.SVC(kernel=rbf, C=5.0, gamma=scale)
Base score source: decision_function_softmax | TTA score source: decision_function_softmax
TTA agreement with base predictions: 0.8107
Predictions generated: 10000 rows
Saved submission: ./submission.csv
Submission columns: ['im_name', 'label']
